In [1]:
import xarray as xr
import rioxarray
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, confusion_matrix
from scipy.ndimage import median_filter, minimum_filter, maximum_filter, uniform_filter
import lightgbm as lgb 
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import joblib
from scipy import stats


In [2]:
payload = joblib.load('trained-ashanti-2021.joblib')
X_train = payload['X_train']
y_train = payload['y_train']
metadata_df_train = payload['metadata_df_train']
X_test = payload['X_test']
y_test = payload['y_test']
metadata_df_test = payload['metadata_df_test']
clf = payload['clf']
user_attrs = payload['user_attrs']

metadata_df_test.head(1)

,lon,lat,fold_idx
542571,-1.384104,7.596895,8549


# Real area from Cocoa area estimation

My issue is that I'm dealing with 2 back-to-back area estimation.
Hence, for each region, I have

1. The real cocoa area
2. The ETHZ's cocoa area estimation
3. My cocoa area estimation

First, I need to figure out how to estimate the real cocoa area from the ETHZ's cocoa area estimation.

## ETHZ's estimate

### Naive Area Estimate

$$A_{ETHZ} = Positive_{ETHZ} = TP_{ETHZ} + FP_{ETHZ}$$

$$Â^{naive}_{ETHZ} = A_{ETHZ}$$


In [3]:
# Cocoa area according to the ETHZ, naively

def compute_naive_area_proportion(y) -> float:
    """Compute the naive area (i.e. pixel counting) given a list of prediction."""
    return float(y.sum() / len(y))

def print_ethz_naive_area_proportion(region: str) -> float:
    # Load the ground truth, as it was split
    payload = joblib.load(f'trained-{region.lower()}-2021.joblib')
    y_test = payload['y_test']

    # Figure out area proportion
    area_proportion = compute_naive_area_proportion(y_test)

    print(f"{region}'s test subregion: ETHZ's cocoa area proportion (naive): {area_proportion * 100:.3f}%")

print_ethz_naive_area_proportion('ashanti')
print_ethz_naive_area_proportion('western north')

ashanti's test subregion: ETHZ's cocoa area proportion (naive): 27.032%
western north's test subregion: ETHZ's cocoa area proportion (naive): 52.203%


### Bias correction

$$A_{ETHZ} = Positive_{ETHZ} = TP_{ETHZ} + FP_{ETHZ} = \frac{TP_{ETHZ}}{Precision_{ETHZ}}$$
$$A_{true} = TP_{ETHZ} + FN_{ETHZ} = \frac{TP_{ETHZ}}{Recall_{ETHZ}}$$

$$\rightarrow A_{true} = A_{ETHZ} * \frac{Precision_{ETHZ}}{Recall_{ETHZ}}$$

$A_{true}$ is the actual cocoa area.\
The thing is, our precision/recall is estimated off some subset (i.e. our test set).\
Hence we make the assumption that our test set's Precision/Recall matches the actual Precision/Recall of our model.

$$\rightarrow Â^{corrected}_{ETHZ} = A_{ETHZ} * \frac{Precision^{test}_{ETHZ}}{Recall^{test}_{ETHZ}}$$

In [4]:
def compute_corrected_area_proportion(y, precision, recall):
    area_proportion = y.sum() / len(y)
    return min(area_proportion * precision / recall, 1.0)

def compute_ethz_corrected_area_proportion(region: str) -> float:
    """Compute the Corrected Area Proportion (CAP) off the ETHZ's prediction, taking only the region's testing area into account."""
    # Load the ground truth, as it was split
    payload = joblib.load(f'trained-{region.lower()}-2021.joblib')
    y_test = payload['y_test']

    # Figure out area proportion
    return compute_corrected_area_proportion(y_test, precision=0.917, recall=0.909) # Precision/Recall from ETHZ study

def print_ethz_corrected_area_proportion(region: str) -> float:
    """Print the Corrected Area Proportion (CAP) off the ETHZ's prediction, taking only the region's testing area into account."""
    area_proportion = compute_ethz_corrected_area_proportion(region)
    print(f"{region}'s test subregion: ETHZ's cocoa area proportion (corrected): {area_proportion * 100:.3f}%")


print_ethz_corrected_area_proportion('ashanti')
print_ethz_corrected_area_proportion('western north')
print_ethz_corrected_area_proportion('western')
print_ethz_corrected_area_proportion('ahafo')
print_ethz_corrected_area_proportion('eastern')

ashanti's test subregion: ETHZ's cocoa area proportion (corrected): 27.270%
western north's test subregion: ETHZ's cocoa area proportion (corrected): 52.663%
western's test subregion: ETHZ's cocoa area proportion (corrected): 40.960%
ahafo's test subregion: ETHZ's cocoa area proportion (corrected): 39.702%
eastern's test subregion: ETHZ's cocoa area proportion (corrected): 13.431%


## Our estimate

The ETHZ's estimated area are the best we'll get.
We make the assumption that the ETHZ's estimate is correct, i.e

$$Â^{corrected}_{ETHZ} = A_{true}$$

From them, we'll compare our estimate, i.e.

$$A_{true} = Â^{corrected}_{ETHZ} = A_{LGBM} * \frac{Precision^{Val}_{LGBM}}{Recall^{Val}_{LGBM}}$$
$$\rightarrow Â^{corrected}_{LGBM} = A_{LGBM} * \frac{Precision^{Val}_{LGBM}}{Recall^{Val}_{LGBM}}$$

We'll measure how far off $Â^{corrected}_{ETHZ}$ we are

In [5]:
def compute_a(y):
    return y.sum() / len(y)

def compute_ahat(yhat, y_vals, yhat_vals):
    area_proportion_1 = yhat.sum() / len(yhat)
    area_proportion_0 = 1 - area_proportion_1
    
    # Figure out precision/false-omission-rate as the weighted-mean
    cms = [confusion_matrix(y_val, yhat_val, labels=[True, False]) for y_val, yhat_val in zip(y_vals, yhat_vals)] # (tp, fn), (fp, tn)
    tps = [e[0][0] for e in cms]
    fns = [e[0][1] for e in cms]
    tns = [e[1][1] for e in cms]
    fps = [e[1][0] for e in cms]

    # Figure out weighted-averaged TN, ...
    n_vals = np.array([len(e) for e in y_vals])
    tp = np.mean(tps)
    fn = np.mean(fns)
    fp = np.mean(fps)
    tn = np.mean(tns)

    # Compute validation set's metrics
    false_omission_rate = fn / (fn + tn)
    precision = tp / (tp + fp)

    # Compute adjusted unbiased area proportion
    ahat = (area_proportion_1 * precision) + (area_proportion_0 * false_omission_rate)

    # Compute Confidence Interval (Olofsson Variance)
    n_cocoa = tp + fp
    n_non_cocoa = fn + tn

    # Variance contribution from both strata
    var_cocoa = (area_proportion_1 ** 2) * (precision * (1 - precision)) / (n_cocoa - 1)
    var_non_cocoa = (area_proportion_0 ** 2) * (false_omission_rate * (1 - false_omission_rate)) / (n_non_cocoa - 1)
    
    # Standard Error
    se = np.sqrt(var_cocoa + var_non_cocoa)
    
    # Margin of Error for 95% Confidence Interval (Z-score = 1.96)
    margin_of_error = 1.96 * se
    
    ci_lower = max(0.0, ahat - margin_of_error) # Clamp at 0%
    ci_upper = min(1.0, ahat + margin_of_error) # Clamp at 100%

    return ahat, (ci_lower, ci_upper)

In [7]:
def print_our_corrected_area_proportion(region: str) -> float:
    """Print the CAP with a Bootstrapped Confidence Interval capturing both data and seed variance."""
    # Load payload
    payload = joblib.load(f'trained-{region.lower()}-2021.joblib')
    X_test = payload['X_test']
    y_test = payload['y_test']
    user_attrs = payload['user_attrs']
    clf = payload['clf']
    yhat_test = clf.predict(X_test)
    
    # Validation-inferred correction
    yhat_vals = [np.array(e) for e in user_attrs['yhat_vals']]
    y_vals = [np.array(e) for e in user_attrs['y_vals']]

    ahat, (ci_lower, ci_upper) = compute_ahat(yhat_test, y_vals, yhat_vals)
    a = compute_a(y_test)
    print(f"{region}'s test estimated CAP: {ahat*100:.2f}% ({100 * ahat / a - 100:.2f}% off ETHZ's estimate) [{ci_lower*100:.2f}%, {ci_upper*100:.2f}%]")
    

# Execution
print_our_corrected_area_proportion('ashanti')
print_our_corrected_area_proportion('western north')
print_our_corrected_area_proportion('western')
print_our_corrected_area_proportion('ahafo')
print_our_corrected_area_proportion('central')
print_our_corrected_area_proportion('eastern')

ashanti's test estimated CAP: 25.42% (3.92% off ETHZ's estimate) [24.58%, 26.25%]
western north's test estimated CAP: 51.33% (0.84% off ETHZ's estimate) [49.48%, 53.17%]
western's test estimated CAP: 41.58% (-0.11% off ETHZ's estimate) [39.97%, 43.18%]
ahafo's test estimated CAP: 41.31% (1.14% off ETHZ's estimate) [39.28%, 43.34%]
central's test estimated CAP: 35.45% (0.59% off ETHZ's estimate) [33.72%, 37.17%]
eastern's test estimated CAP: 13.50% (-8.46% off ETHZ's estimate) [12.65%, 14.36%]
